# 🔬 TB & MS Prediction — Exploratory Data Analysis

This notebook explores the TB (Chest X-Ray) and MS (Brain MRI) datasets before model training.

**What we'll cover:**
1. Dataset overview & class distribution
2. Sample image visualization
3. Pixel intensity analysis
4. Image size distribution
5. Augmentation preview

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import defaultdict
from tqdm.notebook import tqdm

# Add project root
sys.path.insert(0, os.path.dirname(os.getcwd()))
from config import DISEASE_CONFIG, DATA_DIR, TB_DATA_DIR, MS_DATA_DIR

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')
print('✅ Imports done!')

## 1. Dataset Overview

In [ ]:
def count_images(base_dir, classes):
    """Count images in each split."""
    records = []
    for split in ['train', 'val', 'test']:
        for cls in classes:
            path = os.path.join(base_dir, split, cls)
            if os.path.exists(path):
                n = len([f for f in os.listdir(path)
                          if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            else:
                n = 0
            records.append({'Split': split, 'Class': cls, 'Count': n})
    return pd.DataFrame(records)

# TB Dataset
tb_df = count_images(TB_DATA_DIR, ['tb', 'normal'])
print('=== TB Dataset ===')
print(tb_df.pivot(index='Split', columns='Class', values='Count'))

# MS Dataset
ms_df = count_images(MS_DATA_DIR, ['ms', 'normal'])
print('\n=== MS Dataset ===')
print(ms_df.pivot(index='Split', columns='Class', values='Count'))

## 2. Class Distribution Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df, title, classes) in zip(
    axes,
    [
        (tb_df, '🫁 Tuberculosis Dataset', ['tb', 'normal']),
        (ms_df, '🧠 Multiple Sclerosis Dataset', ['ms', 'normal'])
    ]
):
    pivot = df.pivot(index='Split', columns='Class', values='Count')
    pivot.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'],
               edgecolor='black', linewidth=0.8)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Split', fontsize=11)
    ax.set_ylabel('Number of Images', fontsize=11)
    ax.legend(title='Class', fontsize=10)
    ax.tick_params(axis='x', rotation=0)
    for container in ax.containers:
        ax.bar_label(container, fontsize=9)

plt.suptitle('Dataset Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Sample Image Visualization

In [ ]:
def show_samples(base_dir, classes, title, n=4):
    fig, axes = plt.subplots(len(classes), n, figsize=(n*3, len(classes)*3.5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    colors = ['#e74c3c', '#2ecc71']
    
    for row, (cls, color) in enumerate(zip(classes, colors)):
        cls_dir = os.path.join(base_dir, 'train', cls)
        if not os.path.exists(cls_dir):
            continue
        images = [f for f in os.listdir(cls_dir)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:n]
        
        for col in range(n):
            ax = axes[row, col] if len(classes) > 1 else axes[col]
            if col < len(images):
                img = Image.open(os.path.join(cls_dir, images[col])).convert('RGB')
                ax.imshow(img, cmap='gray')
                if col == 0:
                    ax.set_ylabel(cls.upper(), color=color,
                                  fontsize=13, fontweight='bold')
            else:
                ax.axis('off')
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(2.5)
    
    plt.tight_layout()
    plt.show()

# Show TB samples
show_samples(TB_DATA_DIR, ['tb', 'normal'], '🫁 TB Dataset — Sample Chest X-Rays')

# Show MS samples
show_samples(MS_DATA_DIR, ['ms', 'normal'], '🧠 MS Dataset — Sample Brain MRI Scans')

## 4. Pixel Intensity Analysis

In [ ]:
def compute_mean_pixel(base_dir, cls, n_samples=50):
    """Compute mean pixel intensities for a class."""
    cls_dir = os.path.join(base_dir, 'train', cls)
    if not os.path.exists(cls_dir):
        return np.zeros(256)
    files = [f for f in os.listdir(cls_dir)
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:n_samples]
    
    histograms = []
    for fname in files:
        img = Image.open(os.path.join(cls_dir, fname)).convert('L').resize((224, 224))
        arr = np.array(img)
        hist, _ = np.histogram(arr.flatten(), bins=256, range=(0, 255))
        histograms.append(hist)
    return np.mean(histograms, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (base_dir, classes, title) in zip(
    axes,
    [
        (TB_DATA_DIR, ['tb', 'normal'], '🫁 TB X-Ray Pixel Distribution'),
        (MS_DATA_DIR, ['ms', 'normal'], '🧠 MS MRI Pixel Distribution')
    ]
):
    colors = ['#e74c3c', '#3498db']
    for cls, color in zip(classes, colors):
        hist = compute_mean_pixel(base_dir, cls)
        ax.plot(range(256), hist, color=color, linewidth=2, label=cls.upper(), alpha=0.8)
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Pixel Intensity (0=Black, 255=White)', fontsize=11)
    ax.set_ylabel('Mean Frequency', fontsize=11)
    ax.legend(fontsize=10)

plt.suptitle('Pixel Intensity Distribution by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Augmentation Preview

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from config import AUGMENTATION

# Pick any sample image
sample_cls_dir = os.path.join(TB_DATA_DIR, 'train', 'tb')
if os.path.exists(sample_cls_dir):
    sample_img_path = os.path.join(sample_cls_dir,
                                    os.listdir(sample_cls_dir)[0])
    img = load_img(sample_img_path, target_size=(224, 224))
    img_array = img_to_array(img)
    img_array = img_array.reshape((1,) + img_array.shape)

    datagen = ImageDataGenerator(**AUGMENTATION)

    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    fig.suptitle('Data Augmentation Preview — 10 Variations of Same Image',
                 fontsize=13, fontweight='bold')

    axes[0, 0].imshow(img)
    axes[0, 0].set_title('Original', fontweight='bold', color='#2563EB')
    axes[0, 0].axis('off')

    i = 1
    for batch in datagen.flow(img_array, batch_size=1, seed=42):
        if i >= 10:
            break
        row, col = divmod(i, 5)
        axes[row, col].imshow(batch[0].astype(np.uint8))
        axes[row, col].set_title(f'Augmented {i}')
        axes[row, col].axis('off')
        i += 1

    plt.tight_layout()
    plt.show()
else:
    print('⚠️  No images found. Please set up the dataset first.')
    print('   Run: python data/download_data.py')

## ✅ EDA Complete!

Key observations:
- Dataset class distribution
- Visual differences between positive/negative cases
- Pixel intensity patterns per class
- Augmentation applied to training images

**Next Step:** Run `02_Model_Training.ipynb` to train the models.